Iniciamos importando as bibliotecas e funções para manipular os dados, testar as combinações, extrair a matriz de confusão e o relatório completo das métricas.

In [41]:
import pandas as pd # para manipular dados em formato de tabelas dataframes
import time # usada para medir o tempo calculando o custo computacional
from sklearn.ensemble import RandomForestClassifier # algoritmo do random forest
from sklearn.model_selection import GridSearchCV # para testar as combinações dos hiperparâmetros
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score # funções para extrair a matriz de confusão, o relatório das métricas e a acurácia

Abrindo e carregando os dados dos arquivos iris_treino e iris_teste em .csv, verificando também se há valores ausentes.

In [37]:
treino = pd.read_csv('iris_treino.csv')
teste = pd.read_csv('iris_teste.csv')

# verificação de valores ausentes
print("Valores ausentes no treino:")
print(treino.isnull().sum())

print("\nValores ausentes no teste:")
print(teste.isnull().sum())

Valores ausentes no treino:
sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
class           0
dtype: int64

Valores ausentes no teste:
sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
class           0
dtype: int64


Separando as características das espécies da coluna "class" para que o Random Forest possa aprender.

In [5]:
X_treino = treino.drop(columns=['class']) # armazena as medidas das pétalas e sépalas excluindo a coluna 'class' da tabela treino
y_treino = treino['class'] # isola e armazena apenas a coluna 'class' da tabela treino
X_teste = teste.drop(columns=['class']) # armazena as medidas das pétalas e sépalas excluindo a coluna 'class' da tabela teste
y_teste = teste['class'] # isola e armazena apenas a coluna 'class' da tabela teste

O X é maiúsculo porque representa uma matriz matemática que guarda apenas as colunas com as características físicas das flores, como por exemplo o comprimento da sépala, a largura da sépala, o comprimento da pétala e a largura da pétala. Essas são as pistas, que o algoritmo vai usar para tentar adivinhar o tipo de flor.

Já o y, é minúsculo porque representa uma única coluna, onde guarda o "gabarito", as respostas com os nomes das espécies, como por exemplo: Setosa, Versicolor, Virginica, etc.

Em seguida, criamos um dicionário com 'param_classificador' para treinar o classificador, já que a configuração default do Random Forest não é a melhor alternativa para o Iris Dataset que estamos utilizando.

In [18]:
# definindo o modelo de hiperparâmetros para testar variações
param_classificador = {
    'n_estimators': [10, 30, 50],  # quantidade de árvores na floresta
    'max_depth': [3, 5, None]      # profundidade máxima das árvores
}

Em 'n_estimators' testamos a floresta randomica, funcionando com 10 árvores, depois regular para 30 árvores e em seguida, testar com 50 árvores.

O 'max_depth' define a altura dessas árvores, testando o limite com 3 níveis de altura, 5 níveis de altura e None, deixando a árvore crescer livremente até onde precisar.

Logo iniciamos o cronômetro usando a biblioteca time para medir o custo computacional do treinamento do algoritmo.

In [19]:
# iniciando o cronômetro para medir o custo computacional do treino
inicio = time.time()

# treinando o classificador Random Forest
classificador = GridSearchCV(RandomForestClassifier(random_state=42), param_classificador, cv=3)
classificador.fit(X_treino, y_treino)

# calculando o tempo total gasto no treinamento
tempo_execucao = time.time() - inicio

Aqui definimos o algoritmo de Floresta Aleatória que será treinado com a base de treino.

O random_state=42 está dentro, pois o Random Forest cria várias árvores sorteando amostras de dados. Para garantir a reprodução, a semente 42 trava esse sorteio, junto com o dicionário, que contém as opções de quantidade de árvores [10, 30, 50] e profundidades [3, 5, None], e cv=3 que significa validação cruzada em 3 partes, para cada uma das 9 combinações, o GridSearchCV pega a sua base iris_treino.csv e divide matematicamente em 3 partes iguais

Toda vez que o código rodar, o RandomForestClassifier, vai construir exatamente as mesmas árvores, com as mesmas regras, sem depender da sorte, finalizando com o tempo total gasto do treinamento do algoritmo.

A variável classificador que é o objeto do GridSearchCV. O comando '.best_estimator_' vai lá dentro, pega o algoritmo que ganhou com um resultado maior de acurácia no treino e salva na variável 'final'.

Esse comando vai printar na tela mostrando as configurações do modelo vencedor. Ele aprende regras de decisão. O print do objeto 'final' mostra os hiperparâmetros escolhidos, como n_estimators e max_depth.

In [46]:
# armazenando o melhor modelo encontrado pelo GridSearchCV
final = classificador.best_estimator_

# visualizando o modelo
print("Modelo Random Forest:")
print(final)

Modelo Random Forest:
RandomForestClassifier(max_depth=3, n_estimators=10, random_state=42)


Por fim, tentamos prever as classes da base de teste e exibimos os resultados. O '.best_params_' busca no objeto quais foram os números de árvores e profundidades que venceram a validação cruzada. E usamos tempo_execucao para printar o tempo do treinamento do classificador.

In [49]:
# realizando as previsões na base de teste isolada
previsoes = final.predict(X_teste)

# saída de dados
print(f"\nMelhores hiperparâmetros encontrados: {classificador.best_params_}")
print(f"Tempo de treino: {tempo_execucao:.5f} segundos")

# a acurácia
acuracia = accuracy_score(y_teste, previsoes)
print(f"Acurácia: {acuracia:.5f}\n")

# matriz de confusão
matriz = confusion_matrix(y_teste, previsoes)
print("----- Matriz de confusão -----")
cm_t = matriz.T
print(f"Precisão \t Iris-setosa \t Iris-versicolor \t Iris-virginica")
print(f"Iris-setosa \t\t {cm_t[0][0]} \t\t {cm_t[0][1]} \t\t\t {cm_t[0][2]}")
print(f"Iris-versicolor \t {cm_t[1][0]} \t\t {cm_t[1][1]} \t\t\t {cm_t[1][2]}")
print(f"Iris-virginica \t\t {cm_t[2][0]} \t\t {cm_t[2][1]} \t\t\t {cm_t[2][2]}")

# relatório geral
print("\n----- Relatório de classificação por classe e média macro -----")
relatorio_texto = classification_report(y_teste, previsoes)
print(relatorio_texto)


Melhores hiperparâmetros encontrados: {'max_depth': 3, 'n_estimators': 10}
Tempo de treino: 1.51485 segundos
Acurácia: 0.91111

----- Matriz de confusão -----
Precisão 	 Iris-setosa 	 Iris-versicolor 	 Iris-virginica
Iris-setosa 		 15 		 0 			 0
Iris-versicolor 	 0 		 14 			 3
Iris-virginica 		 0 		 1 			 12

----- Relatório de classificação por classe e média macro -----
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        15
Iris-versicolor       0.82      0.93      0.88        15
 Iris-virginica       0.92      0.80      0.86        15

       accuracy                           0.91        45
      macro avg       0.92      0.91      0.91        45
   weighted avg       0.92      0.91      0.91        45



Finalizamos pegando o gabarito com as espécies (y_teste) para confrontar com os palpites que o modelo gerou (previsoes).

Na parte da matriz de confusão vai mostrar o cruzamento de erros e acertos usando o atributo '.T' para fazer uma trasposição de matriz pegando todas as linhas e transformando-as em colunas.

Já o relatório de classificação calcula as métricas: Precisão, Recall e F1-Score, para cada uma das três espécies e a média macro geral dessas métricas.